# Week 4: Autoencoder Training

This notebook trains fully connected autoencoders on Fashion-MNIST, plots reconstruction results, and saves latent representations.

## 1. Imports, paths, and device

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "week1" / "week1_ml_basics.ipynb").is_file()
                     and (folder / "week5" / "week5_vae.ipynb").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter in the research project folder or one of its subfolders.")
WEEK4_ROOT = PROJECT_ROOT / "week4"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = WEEK4_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


## 2. Load a reproducible Fashion-MNIST subset

In [ ]:
transform = transforms.ToTensor()
full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transform)
full_test = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transform)
rng = np.random.default_rng(seed)
train_indices = rng.choice(len(full_train), size=20000, replace=False)
test_indices = rng.choice(len(full_test), size=10000, replace=False)
train_dataset = Subset(full_train, train_indices)
test_dataset = Subset(full_test, test_indices)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
print(len(train_dataset), len(test_dataset))


## 3. Define the autoencoder

The architecture is 784 → 256 → 64 → latent → 64 → 256 → 784.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid()
        )
    def encode(self, x):
        return self.encoder(x)
    def forward(self, x):
        z = self.encode(x)
        return self.decoder(z).view(-1, 1, 28, 28)

print("Done.")


## 4. Train one autoencoder

In [ ]:
def train_autoencoder(latent_dim, epochs=5):
    torch.manual_seed(seed)
    model = Autoencoder(latent_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    history = []
    for epoch in range(epochs):
        model.train(); total = 0.0; count = 0
        for images, _ in train_loader:
            images = images.to(device)
            optimizer.zero_grad()
            reconstruction = model(images)
            loss = loss_fn(reconstruction, images)
            loss.backward(); optimizer.step()
            total += loss.item() * images.size(0); count += images.size(0)
        model.eval(); test_total = 0.0; test_count = 0
        with torch.inference_mode():
            for images, _ in test_loader:
                images = images.to(device)
                test_total += loss_fn(model(images), images).item() * images.size(0)
                test_count += images.size(0)
        history.append({"train_loss": total / count, "test_loss": test_total / test_count})
        print(f"Epoch {epoch + 1}: train loss={history[-1]['train_loss']:.4f}, test loss={history[-1]['test_loss']:.4f}")
    return model, history

model_20, history_20 = train_autoencoder(20)


## 5. Plot training loss

In [ ]:
plt.plot([x["train_loss"] for x in history_20], marker="o", label="Train")
plt.plot([x["test_loss"] for x in history_20], marker="o", label="Test")
plt.xlabel("Epoch"); plt.ylabel("MSE reconstruction loss"); plt.legend(); plt.grid(True); plt.show()


## 6. Show original and reconstructed images

In [ ]:
model_20.eval()
images, _ = next(iter(DataLoader(test_dataset, batch_size=8, shuffle=False)))
with torch.inference_mode():
    reconstructions = model_20(images.to(device)).cpu()
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(images[i, 0], cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(reconstructions[i, 0], cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original"); axes[1, 0].set_ylabel("Reconstructed")
plt.tight_layout(); plt.show()


## 7. Extract and save the latent representation

In [ ]:
latent_batches = []
labels = []
model_20.eval()
with torch.inference_mode():
    for images, batch_labels in test_loader:
        latent_batches.append(model_20.encode(images.to(device)).cpu())
        labels.append(batch_labels)
Z_ae = torch.cat(latent_batches).numpy()
y_test = torch.cat(labels).numpy()
np.save(RESULTS_DIR / "ae_latent_20.npy", Z_ae)
np.save(RESULTS_DIR / "test_labels.npy", y_test)
print("Latent shape:", Z_ae.shape)
